In [3]:
import cv2
import numpy as np

# 설정값
RADIUS = 5  # 점의 크기 (클릭 감지 범위)

src_pts = []
drag_idx = -1  # 드래그 중인 점의 인덱스 (-1은 없음)

def mouse_handler(event, x, y, flags, param):
    global src_pts, drag_idx, img_display

    # 1. 점 찍기 (4개 미만일 때)
    if event == cv2.EVENT_LBUTTONDOWN and len(src_pts) < 4:
        src_pts.append([x, y])
        draw_and_show()

    # 2. 드래그 시작 (4개 다 찍힌 상태에서 점을 클릭했을 때)
    elif event == cv2.EVENT_LBUTTONDOWN and len(src_pts) == 4:
        for i, pt in enumerate(src_pts):
            if np.linalg.norm(np.array(pt) - [x, y]) < RADIUS:
                drag_idx = i
                break

    # 3. 드래그 중
    elif event == cv2.EVENT_MOUSEMOVE and drag_idx != -1:
        src_pts[drag_idx] = [x, y]
        draw_and_show()

    # 4. 드래그 종료
    elif event == cv2.EVENT_LBUTTONUP:
        drag_idx = -1


# 1. 찍은 점들 사이의 거리를 계산해서 실제 비율 찾기
# 좌표 순서: 0:좌상, 1:우상, 2:우하, 3:좌하

# 가로 길이 계산 (상단 두 점 사이 거리와 하단 두 점 사이 거리 중 큰 값)
width_top = np.sqrt(((src_pts[1][0] - src_pts[0][0]) ** 2) + ((src_pts[1][1] - src_pts[0][1]) ** 2))
width_bottom = np.sqrt(((src_pts[2][0] - src_pts[3][0]) ** 2) + ((src_pts[2][1] - src_pts[3][1]) ** 2))
width = int(max(width_top, width_bottom))

# 세로 길이 계산 (좌측 두 점 사이 거리와 우측 두 점 사이 거리 중 큰 값)
height_left = np.sqrt(((src_pts[3][0] - src_pts[0][0]) ** 2) + ((src_pts[3][1] - src_pts[0][1]) ** 2))
height_right = np.sqrt(((src_pts[2][0] - src_pts[1][0]) ** 2) + ((src_pts[2][1] - src_pts[1][1]) ** 2))
height = int(max(height_left, height_right))



def draw_and_show():
    img_draw = origin_img.copy()
    
    # 점과 선 그리기
    if len(src_pts) > 0:
        for i, pt in enumerate(src_pts):
            cv2.circle(img_draw, tuple(pt), RADIUS, (0, 0, 255), -1)
            cv2.putText(img_draw, str(i+1), (pt[0]+10, pt[1]-10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
        
        # 4개가 찍히면 사각형 연결
        if len(src_pts) == 4:
            pts_array = np.array(src_pts, np.int32)
            cv2.polylines(img_draw, [pts_array], True, (0, 255, 0), 2)
            cv2.putText(img_draw, "Press 'Enter' to Transform", (20, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    cv2.imshow('Select & Drag', img_draw)

def transform():
    if len(src_pts) != 4: return
    
    src = np.array(src_pts, dtype=np.float32)
    dst = np.array([[0, 0], [WIDTH, 0], [WIDTH, HEIGHT]], dtype=np.float32) # (예시용, 실제론 4점)
    dst = np.array([[0, 0], [WIDTH, 0], [WIDTH, HEIGHT], [0, HEIGHT]], dtype=np.float32)

    # 1. 고화질 보간법(INTER_CUBIC) 적용해서 변환
    matrix = cv2.getPerspectiveTransform(src, dst)
    res = cv2.warpPerspective(origin_img, matrix, (WIDTH, HEIGHT), flags=cv2.INTER_CUBIC)

    # 2. 샤프닝 필터 적용 (언샤프 마스크 기법)
    # 가우시안 블러로 살짝 흐린 이미지를 만든 뒤 원본에서 빼서 경계선만 추출함
    blurred = cv2.GaussianBlur(res, (0, 0), 3)
    res_sharp = cv2.addWeighted(res, 1.5, blurred, -0.5, 0)

    # 비교를 위해 두 개 다 보여드림
    cv2.imshow('Standard (Cubic)', res)
    cv2.imshow('Sharpened (High Contrast)', res_sharp)
    
    # 만약 흑백 문서처럼 만들고 싶다면 아래 한 줄 추가 (임계값 처리)
    # _, res_bin = cv2.threshold(cv2.cvtColor(res_sharp, cv2.COLOR_BGR2GRAY), 127, 255, cv2.THRESH_BINARY)
    # cv2.imshow('Binary (Document Scan)', res_bin)

# 실행부
origin_img = cv2.imread('news.jpg')
if origin_img is None:
    print("이미지를 불러올 수 없습니다.")
    exit()

cv2.imshow('Select & Drag', origin_img)
cv2.setMouseCallback('Select & Drag', mouse_handler)

print("1. 4개의 점을 시계방향으로 찍으세요.")
print("2. 점을 드래그해서 위치를 수정하세요.")
print("3. 'Enter'를 누르면 변환됩니다. 'r'을 누르면 초기화됩니다.")

while True:
    key = cv2.waitKey(1) & 0xFF
    if key == 13:  # Enter 키
        transform()
    elif key == ord('r'):  # r 키로 초기화
        src_pts = []
        cv2.imshow('Select & Drag', origin_img)
    elif key == 27:  # ESC 키로 종료
        break

cv2.destroyAllWindows()

IndexError: list index out of range